In [8]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
from datetime import date, timedelta

HEADERS = {"User-Agent": "Mozilla/5.0"}

COUNTRIES = {
    "United States": "us",
    "Brazil": "br",
    "India": "in",
    "United Kingdom": "gb",
    "Mexico": "mx"
}

# ----------------------------
# 1️⃣ GENERATE ALL FRIDAYS OF 2025
# ----------------------------
def all_fridays(year):
    d = date(year, 1, 1)
    while d.weekday() != 4:  # Friday
        d += timedelta(days=1)
    fridays = []
    while d.year == year:
        fridays.append(d.strftime("%Y-%m-%d"))
        d += timedelta(days=7)
    return fridays

WEEKS = all_fridays(2025)

rows = []

# ----------------------------
# 2️⃣ SCRAPE WEEKLY DATA (TOP 5 COUNTRIES)
# ----------------------------
for country, code in COUNTRIES.items():
    print(f"\n🌍 Processing {country}")

    for week in WEEKS:
        url = f"https://kworb.net/spotify/country/{code}_weekly.html?date={week}"
        print(f"  ⏳ Week {week}")

        html = requests.get(url, headers=HEADERS).text
        soup = BeautifulSoup(html, "html.parser")

        table = soup.find("table")
        if not table:
            continue

        for tr in table.find_all("tr")[1:]:
            tds = tr.find_all("td")
            if len(tds) < 3:
                continue

            artist_title = tds[2].text.strip()
            if " - " in artist_title:
                artist, track = artist_title.split(" - ", 1)
            else:
                artist, track = "Unknown", artist_title

            # Robust numeric detection
            streams = None
            for td in tds:
                txt = td.text.replace(",", "").strip()
                if re.fullmatch(r"\d+", txt):
                    val = int(txt)
                    if streams is None or val > streams:
                        streams = val

            if streams is None:
                continue

            rows.append([
                country,
                week,
                track.strip(),
                artist.strip(),
                streams
            ])

# ----------------------------
# 3️⃣ WEEKLY → MONTHLY AGGREGATION
# ----------------------------
df = pd.DataFrame(
    rows,
    columns=["country", "week", "track", "artist", "weekly_streams"]
)

df["week"] = pd.to_datetime(df["week"])
df["month"] = df["week"].dt.to_period("M")

monthly = (
    df.groupby(["country", "month", "track", "artist"])["weekly_streams"]
    .sum()
    .reset_index()
    .rename(columns={"weekly_streams": "monthly_streams"})
)

# ----------------------------
# 4️⃣ SAVE STEP-1 OUTPUT
# ----------------------------
monthly.to_csv(
    "top5_countries_monthly_2025.csv",
    index=False
)

print("\n🎉 STEP-1 COMPLETED")
print("📁 File created: top5_countries_monthly_2025.csv")



🌍 Processing United States
  ⏳ Week 2023-01-06
  ⏳ Week 2023-01-13
  ⏳ Week 2023-01-20
  ⏳ Week 2023-01-27
  ⏳ Week 2023-02-03
  ⏳ Week 2023-02-10
  ⏳ Week 2023-02-17
  ⏳ Week 2023-02-24
  ⏳ Week 2023-03-03
  ⏳ Week 2023-03-10
  ⏳ Week 2023-03-17
  ⏳ Week 2023-03-24
  ⏳ Week 2023-03-31
  ⏳ Week 2023-04-07
  ⏳ Week 2023-04-14
  ⏳ Week 2023-04-21
  ⏳ Week 2023-04-28
  ⏳ Week 2023-05-05
  ⏳ Week 2023-05-12
  ⏳ Week 2023-05-19
  ⏳ Week 2023-05-26
  ⏳ Week 2023-06-02
  ⏳ Week 2023-06-09
  ⏳ Week 2023-06-16
  ⏳ Week 2023-06-23
  ⏳ Week 2023-06-30
  ⏳ Week 2023-07-07
  ⏳ Week 2023-07-14
  ⏳ Week 2023-07-21
  ⏳ Week 2023-07-28
  ⏳ Week 2023-08-04
  ⏳ Week 2023-08-11
  ⏳ Week 2023-08-18
  ⏳ Week 2023-08-25
  ⏳ Week 2023-09-01
  ⏳ Week 2023-09-08
  ⏳ Week 2023-09-15
  ⏳ Week 2023-09-22
  ⏳ Week 2023-09-29
  ⏳ Week 2023-10-06
  ⏳ Week 2023-10-13
  ⏳ Week 2023-10-20
  ⏳ Week 2023-10-27
  ⏳ Week 2023-11-03
  ⏳ Week 2023-11-10
  ⏳ Week 2023-11-17
  ⏳ Week 2023-11-24
  ⏳ Week 2023-12-01
  ⏳ Week 202

In [1]:
import pandas as pd
df = pd.read_csv("top5_countries_monthly_2023.csv")

print(df["country"].value_counts())
print(df["month"].unique())


country
Brazil            2400
India             2400
Mexico            2400
United Kingdom    2400
United States     2400
Name: count, dtype: int64
['2023-01' '2023-02' '2023-03' '2023-04' '2023-05' '2023-06' '2023-07'
 '2023-08' '2023-09' '2023-10' '2023-11' '2023-12']


track                   artist           
No Role Modelz          J. Cole              64029302168
Sweater Weather         The Neighbourhood    63032269196
Blinding Lights         The Weeknd           59570769700
Mr. Brightside          The Killers          57119948132
Dreams - 2004 Remaster  Fleetwood Mac        49230545624
Name: monthly_streams, dtype: int64


                       track             artist  yearly_streams
494           No Role Modelz            J. Cole     64029302168
690          Sweater Weather  The Neighbourhood     63032269196
89           Blinding Lights         The Weeknd     59570769700
461           Mr. Brightside        The Killers     57119948132
196   Dreams - 2004 Remaster      Fleetwood Mac     49230545624
681             Stick Season         Noah Kahan     49158832632
675   Starboy (w/ Daft Punk)         The Weeknd     48963576376
231                 Espresso  Sabrina Carpenter     48915188816
670  Something in the Orange         Zach Bryan     48058388456
13                       505     Arctic Monkeys     46962971328


            country                                              track  \
91           Brazil       Lapada Dela - Ao Vivo (w/ Matheus Fernandes)   
63           Brazil                             Erro Gostoso - Ao Vivo   
42           Brazil         Coração Partido (Corazón Partío) - Ao Vivo   
199          Brazil                           Última Saudade - Ao Vivo   
175          Brazil                                 Tubarões - Ao Vivo   
214           India  Agar Tum Saath Ho (From "Tamasha") (w/ Arijit ...   
383           India  Tujhe Kitna Chahne Lage (From "Kabir Singh") (...   
253           India                Excuses (w/ Gurinder Gill, Intense)   
357           India                           Shayad (w/ Arijit Singh)   
296           India                     Kaise Hua (From "Kabir Singh")   
471          Mexico                          LA CANCIÓN (w/ Bad Bunny)   
490          Mexico                                          Me Rehúso   
534          Mexico           Que Onda

country
United States     2620236056152
Mexico            1736996074892
India             1405906990644
Brazil             803828734592
United Kingdom     773912054448
Name: monthly_streams, dtype: int64


artist
Junior H         181890101380
The Weeknd       172704030824
Bad Bunny        163743699392
Pritam           145524704520
Morgan Wallen    135856633588
Name: monthly_streams, dtype: int64


                                     track  min_streams  max_streams  \
0                                 /Montate     21443696     26804620   
586                            Ranjheya Ve    424358648    530448310   
575  Qué Onda Perdida (w/ Gerardo Coronel)    414491816    518114770   
576          Qué Pasaría... (w/ Bad Bunny)    826380376   1032975470   
577                               ROCKSTAR   1723805536   2154756920   

     growth_percent  
0              25.0  
586            25.0  
575            25.0  
576            25.0  
577            25.0  
